# 08 — Derating, thermal, component stress

Datasheet: $P = I_{\mathrm{OUT}}(V_{\mathrm{IN}}-V_{\mathrm{OUT}}) + I_{\mathrm{GND}} V_{\mathrm{IN}}$.
θJA from Table 1 vs copper area. SOA at $T_A=25$ and $85\,^\circ\mathrm{C}$.

Horizontal **stress bars** match comparator notebook 06:

| Chart | Axis | Guideline |
|-------|------|-----------|
| Power dissipation | mW | — |
| Power utilization | $P/P_{\mathrm{rated}}$ (%) | 50 % use |
| Voltage utilization | $V / (0.80\,V_{\max})$ (%) | 100 % = 80 % derate |
| Thermal utilization (U1) | $P / P_{\mathrm{Tjmax}}$ (%) | 100 % = $T_j=125\,^\circ\mathrm{C}$ |
| $\Delta T$ | °C | — |

Grouped bars repeat the same metrics at **5.4 / 12 / 24 V** (inside the datasheet 5.4 V to 80 V range)
and **1 / 35 / 50 mA**.

## Setup — editable BOM and run knobs

This cell is shared by every notebook. It puts `P5V_Regulator/` on `sys.path`
and builds **one** `CircuitParams` object. Change a resistor, capacitor, or
tolerance here and re-run: analytical, SPICE, WC, and MC all follow this object.

**Plot callouts:** each figure cell has an `ann = [dict(...), ...]` list.
Edit `text`, `xy` (data coords of the arrow tip), and `offset=(dx, dy)` in
**points** to nudge a label. Use `xytext=(x, y)` instead of `offset` for data
coordinates. `arrow=False` for a plain label. `ann = []` hides all callouts.

Tolerances are **fractions** (0.01 = 1 %). Figures go to `results/figures/`.

In [1]:
from pathlib import Path
import sys
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

def _find_project_root() -> Path:
    start = Path.cwd().resolve()
    named = Path("Sections/Electronics/Analog_and_Digital/LT3010_5V_Supply")
    candidates = [start, *start.parents, start / named]
    if start.name == "notebooks":
        candidates.insert(0, start.parent)
    for cand in candidates:
        if (cand / "p5v_design" / "__init__.py").is_file():
            return cand
    return start

ROOT = _find_project_root()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from p5v_design.params import CircuitParams, ToleranceSpec, format_eng, LdoParams
from p5v_design import analysis, schematic, plots, stability
from p5v_design import plots_wc_mc as pwc
from p5v_design.ldo_model import dropout_v, ignd_a, tj_c, psrr_db, vadj_typical_vs_temp
from dataclasses import replace

# --- EDITABLE BOM ---
# Datasheet TA01, "5V Supply with Shutdown" (LT3010/LT3010-5 Rev. J page 1).
# Fixed 5 V part: SENSE tied to OUT (R1 = 0). No external divider (R2 open).
# Capacitors are the 1 µF parts drawn on that figure. Load is the labeled 50 mA.
# VIN is printed as 5.4 V to 80 V; 12 V is the single-point bias inside that range.
R1 = 0.0             # SENSE shorted to OUT
R2 = 1.0e12          # open — no bottom resistor
CFF = 0.0            # no feedforward capacitor on the fixed part
COUT = 1.0e-6
CIN = 1.0e-6
ESR_OUT = 10e-3      # ceramic assumption, not printed on TA01
ESR_IN = 10e-3
RLOAD = 100.0        # 5 V / 50 mA
R_TOL = 0.01         # 1 %
C_TOL = 0.10         # 10 %
VIN_NOM = 12.0
VIN_MIN = 5.4
VIN_MAX = 80.0
ILOAD_OP = 50e-3
N_MC = 2000
N_SPICE_MC = 40
N_LTSPICE_MC = 200
SKIP_SPICE_WC_MC = False
RUN_LTSPICE_BATCH = False
SPICE_WORKERS = None
LTSPICE_WORKERS = None

p = CircuitParams(
    r1=R1, r2=R2, cff=CFF, cout=COUT, cin=CIN,
    esr_out=ESR_OUT, esr_in=ESR_IN, r_load=RLOAD,
    r_tol=ToleranceSpec(R_TOL), c_tol=ToleranceSpec(C_TOL),
)
p = replace(p, op=replace(p.op, vin_nom=VIN_NOM, vin_min=VIN_MIN, vin_max=VIN_MAX,
                           iload_op=ILOAD_OP, r_load=RLOAD))

FIG = ROOT / "results" / "figures"
FIG.mkdir(parents=True, exist_ok=True)
print("gain =", p.gain, "  Vout nom =", analysis.vout_dc(p))
print("SPICE_WORKERS =", SPICE_WORKERS, "  N_MC =", N_MC)


gain = 1.0   Vout nom = 5.0
SPICE_WORKERS = None   N_MC = 2000


## Power / voltage table at the 12 V / 50 mA bias

Ambient $T_A=25\,^\circ\mathrm{C}$. Status: `OK`, `P>50%`, `V>80%`, `Tj>max`.
CSV: `results/thermal_operate.csv`.

In [2]:
from p5v_design.thermal import power_table, copper_table, power_tables_vs_vin, power_tables_vs_load, bom_only
from p5v_design.derating import power_derating_curve

pt = power_table(p, ta_C=25.0)
bom = bom_only(pt)
display(pt[[
    "ref", "package", "P_mW", "P_rated_W", "utilization_pct",
    "V_applied_V", "V_derated_V", "V_util_derated_pct",
    "P_util_thermal_pct", "dT_C", "Tj_C", "status", "note",
]])
pt.to_csv(ROOT/"results"/"thermal_operate.csv", index=False)
display(copper_table())

,ref,package,P_mW,P_rated_W,utilization_pct,V_applied_V,V_derated_V,V_util_derated_pct,P_util_thermal_pct,dT_C,Tj_C,status,note
0,R1,0805,0.000000e+00,0.125,0.000000e+00,0.0,120.0,0.000000,0.000000e+00,0.000000e+00,25.000,OK,feedback top
1,R2,0805,2.500000e-08,0.125,2.000000e-08,5.0,120.0,4.166667,5.500000e-09,5.500000e-09,25.000,OK,feedback bot
2,Rload,—,2.500000e+02,0.000,NaN,5.0,NaN,NaN,NaN,0.000000e+00,25.000,OK,Iload equivalent (not a board part)
3,U1,MS8E,3.716000e+02,0.000,NaN,12.0,64.0,18.750000,1.486400e+01,1.486400e+01,39.864,OK,Ppass=0.350 W + Pgnd=0.022 W
4,C2,1210,0.000000e+00,0.000,NaN,5.0,12.8,39.062500,NaN,0.000000e+00,25.000,OK,Cout
5,C4,1206,0.000000e+00,0.000,NaN,12.0,20.0,60.000000,NaN,0.000000e+00,25.000,OK,Cin
6,C1,1210,0.000000e+00,0.000,NaN,0.0,12.8,0.000000,NaN,0.000000e+00,25.000,OK,Cff


,topside_mm2,backside_mm2,board_mm2,rth_ja
0,2500.0,2500.0,2500.0,40.0
1,1000.0,2500.0,2500.0,45.0
2,225.0,2500.0,2500.0,50.0
3,100.0,2500.0,2500.0,62.0


## Stress bars at operate — power, voltage, ΔT (comparator 06)

R1/R2 dissipation is tiny (divider ~0.6 mA). **U1** is the thermal part.
Cout/Cin voltage stress uses the 5 V output and the 12 V bias. The datasheet input can be as high as 80 V, so the input capacitor voltage rating is an assumption, not a value printed on the example.

Dummy `Rload` is an Iload equivalent, not a board part — it is omitted from
voltage / utilization charts. Black ticks on the applied-voltage chart are
**80% derated** $V_{\max}$; gray ticks are catalog $V_{\max}$.

In [3]:
ann_p = [dict(text="U1 is the wattage", xy=(float(pt.loc[pt.ref=="U1", "P_mW"].iloc[0]), "U1"), offset=(12, 10), arrow=True)]
plots.plot_stress_bars(
    pt, value_col="P_mW", xlabel="Power (mW)", title="BOM power dissipation at 12 V / 50 mA",
    color="C0", annotations=ann_p, path=FIG/"08_stress_PmW.png",
)
plt.show()
plots.plot_stress_bars(
    bom[bom["P_rated_W"] > 0], value_col="utilization_pct",
    xlabel="P / P_rated (%)", title="Resistor power utilization",
    vline=50, vline_label="50% use guideline", color="C4",
    path=FIG/"08_stress_Putil.png",
)
plt.show()
plots.plot_stress_bars(
    bom[bom["V_max_V"] > 0], value_col="V_util_derated_pct",
    xlabel="V / (0.80 Vmax) (%)", title="Voltage stress vs 80% derated rating",
    vline=100, vline_label="100% of 80% derate",
    vline2=80, vline2_label="80% of derated V (extra margin)",
    color="C1", path=FIG/"08_stress_Vutil.png",
)
plt.show()
plots.plot_applied_voltage(
    bom, title="Applied voltage at 12 V / 50 mA", path=FIG/"08_stress_Vapplied.png",
)
plt.show()
plots.plot_stress_bars(
    bom[bom["dT_C"].notna()], value_col="dT_C",
    xlabel="ΔT (°C)", title="Junction / package temperature rise",
    color="C3", path=FIG/"08_stress_dT.png",
)
plt.show()
plots.plot_stress_bars(
    bom[bom["P_util_thermal_pct"].notna()], value_col="P_util_thermal_pct",
    xlabel="P / P(Tj=125 °C) (%)", title="Thermal-budget utilization",
    vline=100, vline_label="Tjmax", color="C5",
    path=FIG/"08_stress_Pthermal.png",
)
plt.show()
plots.plot_cap_voltage_bars(bom, path=FIG/"08_cap_voltage.png")
plt.show()

cell:6: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


cell:13: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


cell:21: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


cell:25: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


cell:31: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()
cell:38: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()
cell:40: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## Grouped bars across the datasheet input range and vs load

Same idea as comparator 06 (operate / UVLO / full-scale): here **5.4 / 12 / 24 V**
and **1 / 35 / 50 mA**. U1 $P$ and $\Delta T$ move; the divider barely does.

In [4]:
vin_tabs = power_tables_vs_vin(p, ta_C=25.0)
load_tabs = power_tables_vs_load(p, ta_C=25.0)
vin_bom = {k: bom_only(df) for k, df in vin_tabs.items()}
load_bom = {k: bom_only(df) for k, df in load_tabs.items()}
for key, df in vin_tabs.items():
    df.to_csv(ROOT/"results"/f"thermal_{key.replace(' ', '')}.csv", index=False)
plots.plot_stress_grouped(
    vin_tabs, value_col="P_mW", xlabel="Power (mW)",
    title="Power vs Vin (5.4 / 12 / 24 V)", path=FIG/"08_stress_P_vs_vin.png",
)
plt.show()
plots.plot_stress_grouped(
    vin_bom, value_col="utilization_pct", xlabel="P / P_rated (%)",
    title="Power utilization vs Vin", vline=50, vline_label="50% use",
    path=FIG/"08_stress_Putil_vs_vin.png",
)
plt.show()
plots.plot_stress_grouped(
    vin_bom, value_col="V_util_derated_pct", xlabel="V / (0.80 Vmax) (%)",
    title="Voltage stress vs Vin", vline=100, vline_label="100% of 80% derate",
    vline2=80, vline2_label="80% of derated V",
    path=FIG/"08_stress_V_vs_vin.png",
)
plt.show()
plots.plot_applied_voltage_grouped(
    vin_bom, title="Applied voltage vs Vin (5.4 / 12 / 24 V)",
    path=FIG/"08_stress_Vapplied_vs_vin.png",
)
plt.show()
plots.plot_stress_grouped(
    vin_bom, value_col="dT_C", xlabel="ΔT (°C)",
    title="ΔT vs Vin", path=FIG/"08_stress_dT_vs_vin.png",
)
plt.show()
plots.plot_stress_grouped(
    vin_bom, value_col="P_util_thermal_pct", xlabel="P / P(Tj=125 °C) (%)",
    title="Thermal-budget utilization vs Vin", vline=100, vline_label="Tjmax",
    path=FIG/"08_stress_Pthermal_vs_vin.png",
)
plt.show()
plots.plot_stress_grouped(
    load_tabs, value_col="P_mW", xlabel="Power (mW)",
    title="Power vs load (1 / 35 / 50 mA)", path=FIG/"08_stress_P_vs_load.png",
)
plt.show()
plots.plot_stress_grouped(
    load_bom, value_col="utilization_pct", xlabel="P / P_rated (%)",
    title="Power utilization vs load", vline=50, vline_label="50% use",
    path=FIG/"08_stress_Putil_vs_load.png",
)
plt.show()
plots.plot_stress_grouped(
    load_bom, value_col="V_util_derated_pct", xlabel="V / (0.80 Vmax) (%)",
    title="Voltage stress vs load", vline=100, vline_label="80% derate",
    path=FIG/"08_stress_V_vs_load.png",
)
plt.show()
plots.plot_applied_voltage_grouped(
    load_bom, title="Applied voltage vs load (1 / 35 / 50 mA)",
    path=FIG/"08_stress_Vapplied_vs_load.png",
)
plt.show()
plots.plot_stress_grouped(
    load_bom, value_col="dT_C", xlabel="ΔT (°C)",
    title="ΔT vs load", path=FIG/"08_stress_dT_vs_load.png",
)
plt.show()
plots.plot_stress_grouped(
    load_bom, value_col="P_util_thermal_pct", xlabel="P / P(Tj=125 °C) (%)",
    title="Thermal-budget utilization vs load", vline=100, vline_label="Tjmax",
    path=FIG/"08_stress_Pthermal_vs_load.png",
)
plt.show()

cell:11: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()
cell:17: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()
cell:24: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()
cell:29: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


cell:34: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


cell:40: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


cell:45: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


cell:51: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()
cell:57: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()
cell:62: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()
cell:67: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()
cell:73: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## SOA, $T_j$ map, copper area, 0805 derating curve

In [5]:
ann_soa = [dict(text="12 V / 50 mA", xy=(12.0, 35.0), offset=(14, 12), arrow=True, color="w")]
plots.plot_soa(p, ta_c=25.0, annotations=ann_soa, path=FIG/"08_soa_25C.png")
plt.show()
plots.plot_soa(p, ta_c=85.0, annotations=[dict(text="85 °C ambient", xy=(9, 35), offset=(12, 10), arrow=True)],
               path=FIG/"08_soa_85C.png")
plt.show()
ann_map = [dict(text="12 V / 50 mA", xy=(12.0, 50.0), offset=(12, 10), arrow=True, color="w")]
plots.plot_tj_map(p, ta_c=25.0, annotations=ann_map, path=FIG/"08_tj_map.png")
plt.show()
plots.plot_copper_rth(annotations=[dict(text="2500 mm² → 40 °C/W", xy=(2500, 40), offset=(-80, 14), arrow=True)],
                      path=FIG/"08_copper.png")
plt.show()
cur = power_derating_curve(p)
fig, ax = plt.subplots(figsize=(8.0, 3.8))
ax.plot(cur.temp_c, cur.p_rated_derated_W, "k-", lw=1.7, label="rated (derated)")
ax.plot(cur.temp_c, cur.p_use_W, "C3--", lw=1.3, label="50 % use")
ax.set_xlabel("T (°C)"); ax.set_ylabel("W")
ax.set_title("0805 resistor power derating")
ax.legend(fontsize=8); ax.grid(True, alpha=0.3)
plots.savefig(fig, FIG/"08_r_derate.png"); plt.show()

cell:3: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()
p5v_design/plots.py:511: RuntimeWarning: More than 20 figures have been opened. Figures created through the pyplot interface (`matplotlib.pyplot.figure`) are retained until explicitly closed and may consume too much memory. (To control this warning, see the rcParam `figure.max_open_warning`). Consider using `matplotlib.pyplot.close()`.
  fig, ax = plt.subplots(figsize=(8.2, 4.6))


cell:6: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()
cell:9: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


cell:12: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()
cell:20: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plots.savefig(fig, FIG/"08_r_derate.png"); plt.show()


## Ceramic DC-bias (datasheet Figs. 3–4)

A 16 V 10 µF **Y5V** can look like 1–2 µF at 5 V. **X7R/X5R** is required for
Cout. This design uses 10 µF / 5 mΩ — stay on X7R and verify C at 5 V DC.

In [6]:
ann_c = [dict(text="5 V Cout", xy=(5.0, 85), offset=(12, 10), arrow=True)]
plots.plot_ceramic_dc_bias(annotations=ann_c, path=FIG/"08_ceramic_dc_bias.png")
plt.show()

cell:3: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()
